# ntl-glmllpool-glm53

Per-token log-likelihood of every train passage under GLM-5.3, for refitting the likelihood head of the GLM stage offline. The three code files are estopadilla's, from the V1 payload of his public [0.689](https://www.kaggle.com/code/estopadilla/0-689-real-glm-5-3-likelihood-on-2xt4) (Apache 2.0), unchanged; `ntl-segrun` is his build ([ntl-glm-likelihood-train](https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train)); the GGUF segments are jamesliebel's mirrors. Settings as the test-time stage, cut at 160 tokens. The passages are exp053's 2,000 open-pool rows (kaggle/glmll/pool_rows_v17.csv), not the train rows.

In [ ]:
import os
os.makedirs('/kaggle/temp/glmll', exist_ok=True)

In [ ]:
%%writefile /kaggle/temp/glmll/segio.py
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile /kaggle/temp/glmll/features.py
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile /kaggle/temp/glmll/run_pipeline.py
"""Score texts under a big GLM model that ships as layer-segment GGUFs, with ntl-segrun.

Every text goes through segment k before segment k+1 is loaded, so each weight is read once per chunk.
The residual state between segments lives in one file on local disk, rewritten in place.
If the disk cannot hold the state of all texts, the texts are split into chunks (each chunk reads the
model again).

    python run_pipeline.py --bin BIN_DIR --segs 'GLOB' --csv train.csv --out OUT_DIR --tag flash
        [--work /tmp/ntl] [--prefix STR] [--max-text-tokens 320] [--seq-batch 8] [--state-dtype auto]
        [--limit N] [--deadline-s S] [--pack] [--ubatch-div P] [--budget-s S] [--prefetch [--prefetch-after F]]

--pack sorts the texts by token count (longest first) and packs them into padded decodes (see ntl-segrun.cpp);
the scores are put back in the input order. --budget-s is a hard limit for the whole call: every ntl-segrun call
gets the time left as its timeout, and after each segment the time of the rest is projected from the segments
done so far; if it does not fit, the call stops with exit code 3 (BudgetExceeded) and writes nothing.
--prefetch: once ntl-segrun has loaded segment k ("context ready"), a background thread reads segment k+1 once
(to /dev/null), so its pages are in the page cache when it is mapped; during the last segment it reads
--prefetch-after (e.g. the first segment of the next model). The same bytes, only earlier: scores are unchanged.

Writes OUT_DIR/{tag}_scores.npz (per-token logprob/entropy/m2/rank, offsets, ids), {tag}_features.csv
and {tag}_timing.json.
"""
import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import threading
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class BudgetExceeded(RuntimeError):
    """The time budget (--budget-s) is not enough: stop and let the caller fall back."""


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


_PREFETCHED = {}


def prefetch(path):
    """Read a file once in a daemon thread (page cache warm-up for the next mmap); each file at most once."""
    path = os.path.realpath(path)
    if path in _PREFETCHED:
        return
    info = _PREFETCHED[path] = {'s': None}

    def read():
        t0 = time.time()
        with open(path, 'rb', buffering=0) as f:
            try:
                os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
            except (AttributeError, OSError):
                pass
            buf = bytearray(16 << 20)
            while f.readinto(buf):
                pass
        info['s'] = time.time() - t0
    threading.Thread(target=read, daemon=True).start()


def run(cmd, log, timeout=None, on_ready=None):
    """Run one ntl-segrun call; on_ready() is called when it prints "context ready" (the model is loaded)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    expired = threading.Event()

    def kill():
        expired.set()
        p.kill()
    timer = threading.Timer(timeout, kill) if timeout is not None else None
    if timer is not None:
        timer.daemon = True
        timer.start()
    lines = []
    for line in p.stdout:
        lines.append(line)
        if on_ready is not None and 'context ready' in line:
            on_ready()
            on_ready = None
    p.wait()
    if timer is not None:
        timer.cancel()
    out = ''.join(lines)
    if expired.is_set():
        with open(log, 'a') as lf:
            lf.write('$ ' + ' '.join(cmd) + '\n' + out + '\n[killed: timeout]\n')
        raise BudgetExceeded(f'timeout after {timeout:.0f} s: {" ".join(cmd)}')
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + out)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(out.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in out else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='glob of the segment files')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--deadline-s', type=float, default=0, help='stop before starting a chunk after this many seconds')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--budget-s', type=float, default=0, help='hard time limit of the whole call (0 = none)')
    ap.add_argument('--prefetch', action='store_true', help='read the next segment while the current one computes')
    ap.add_argument('--prefetch-after', default='', help='with --prefetch: file to read during the last segment')
    a = ap.parse_args()
    try:
        main_run(a)
    except BudgetExceeded as e:
        print(f'[{a.tag}] BUDGET EXCEEDED: {e}', flush=True)
        sys.exit(3)


def main_run(a):

    t_start = time.time()

    def left():
        return None if not a.budget_s else a.budget_s - (time.time() - t_start)

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(glob.glob(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div, 'budget_s': a.budget_s}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        if a.deadline_s and time.time() - t_start > a.deadline_s:
            print(f'[{a.tag}] deadline reached before chunk {ci}, stopping', flush=True)
            break
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                if left() is not None and left() <= 0:
                    raise BudgetExceeded('no time left')
                nxt = (segs[k + 1] if k + 1 < n_seg else segs[0] if ci + 1 < len(chunks) else a.prefetch_after)
                ready = (lambda f=nxt: prefetch(f)) if a.prefetch and nxt else None
                try:
                    dt = run(cmd, log, timeout=left(), on_ready=ready)
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
            if a.budget_s:
                # projection: the remaining segments of this chunk at the mean so far (the last one, with the
                # softmax over the vocabulary, counts double), plus the other chunks at this chunk's pace
                rest = n_seg - k - 1
                per = float(np.mean(seg_t))
                proj = per * (rest + (1 if rest > 0 else 0)) + per * n_seg * 1.15 * (len(chunks) - ci - 1)
                if proj > left():
                    raise BudgetExceeded(f'projected {proj:.0f} s for the rest, {left():.0f} s left')
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    if a.prefetch:
        timing['prefetch_s'] = {os.path.basename(k): v['s'] for k, v in _PREFETCHED.items()}
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
import glob, json, os, shutil, subprocess, sys, time
TAG, PATTERN, N_SEG, MAX_TEXT_TOKENS = 'glm53', 'glm53_iq3xxs-seg*-of-19-*.gguf', 19, 160
PREFIX = '[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>'   # the test-time stage's
subprocess.run('nvidia-smi -L; nproc; free -g; df -h /kaggle/working /kaggle/temp /tmp', shell=True)
TRAIN = next(p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True) if 'name-that-llm' in p)
segs = {}
for p in sorted(glob.glob('/kaggle/input/**/' + PATTERN, recursive=True)):
    segs.setdefault(os.path.basename(p), p)   # a file may be mounted under two roots
assert len(segs) == N_SEG, sorted(segs)
SEGDIR = f'/kaggle/temp/segs_{TAG}'
os.makedirs(SEGDIR, exist_ok=True)
for name, p in segs.items():
    os.symlink(p, os.path.join(SEGDIR, name))
art = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ntl-bin/ntl-segrun', recursive=True))[0]
for f in ('lib/libcublas.so.13', 'lib/libcublasLt.so.13', 'lib/libcudart.so.13'):
    assert os.path.isfile(os.path.join(art, f)), f
BIN = '/kaggle/temp/ntl-bin'
os.makedirs(BIN, exist_ok=True)
shutil.copyfile(os.path.join(art, 'ntl-segrun'), os.path.join(BIN, 'ntl-segrun'))
os.chmod(os.path.join(BIN, 'ntl-segrun'), 0o755)
env = dict(os.environ, LD_LIBRARY_PATH=os.path.join(art, 'lib') + ':' + os.environ.get('LD_LIBRARY_PATH', ''))
WORK = max(['/kaggle/temp', '/tmp'], key=lambda d: shutil.disk_usage(d).free) + '/glmll-work'
OUT = '/kaggle/working/out'
print(TRAIN, art, WORK, len(segs), 'segments', flush=True)
import pandas as pd
POOL_IDS = ['ntlop_124159', 'ntlop_273248', 'ntlop_290532', 'ntlop_285425', 'ntlop_014225', 'ntlop_042345', 'ntlop_208246', 'ntlop_272530', 'ntlop_278056', 'ntlop_122677', 'ntlop_035013', 'ntlop_237450', 'ntlop_114698', 'ntlop_112468', 'ntlop_001260', 'ntlop_013395', 'ntlop_281667', 'ntlop_021470', 'ntlop_299274', 'ntlop_267461', 'ntlop_010883', 'ntlop_306240', 'ntlop_280340', 'ntlop_210288', 'ntlop_246677', 'ntlop_035722', 'ntlop_227402', 'ntlop_042603', 'ntlop_231598', 'ntlop_020511', 'ntlop_272720', 'ntlop_042831', 'ntlop_280269', 'ntlop_278061', 'ntlop_265641', 'ntlop_126675', 'ntlop_042758', 'ntlop_113506', 'ntlop_132010', 'ntlop_295436', 'ntlop_122419', 'ntlop_034572', 'ntlop_266303', 'ntlop_115423', 'ntlop_237896', 'ntlop_008679', 'ntlop_135477', 'ntlop_128217', 'ntlop_284159', 'ntlop_045114', 'ntlop_209095', 'ntlop_033905', 'ntlop_113604', 'ntlop_263665', 'ntlop_127533', 'ntlop_033433', 'ntlop_297598', 'ntlop_012377', 'ntlop_265620', 'ntlop_209532', 'ntlop_250523', 'ntlop_220107', 'ntlop_294402', 'ntlop_034899', 'ntlop_281389', 'ntlop_253164', 'ntlop_026318', 'ntlop_050723', 'ntlop_291390', 'ntlop_136489', 'ntlop_275702', 'ntlop_026290', 'ntlop_012870', 'ntlop_208496', 'ntlop_136088', 'ntlop_210921', 'ntlop_234177', 'ntlop_273765', 'ntlop_285690', 'ntlop_209345', 'ntlop_033490', 'ntlop_112729', 'ntlop_128388', 'ntlop_284309', 'ntlop_208858', 'ntlop_275764', 'ntlop_208731', 'ntlop_235593', 'ntlop_235002', 'ntlop_234444', 'ntlop_227819', 'ntlop_248189', 'ntlop_274137', 'ntlop_248773', 'ntlop_209409', 'ntlop_278069', 'ntlop_264292', 'ntlop_233979', 'ntlop_020881', 'ntlop_299587', 'ntlop_013998', 'ntlop_249448', 'ntlop_285916', 'ntlop_210703', 'ntlop_227571', 'ntlop_032902', 'ntlop_043759', 'ntlop_211607', 'ntlop_038951', 'ntlop_266254', 'ntlop_126534', 'ntlop_286573', 'ntlop_024642', 'ntlop_041333', 'ntlop_219839', 'ntlop_003946', 'ntlop_277992', 'ntlop_114229', 'ntlop_000358', 'ntlop_112352', 'ntlop_025509', 'ntlop_113651', 'ntlop_050075', 'ntlop_113772', 'ntlop_218425', 'ntlop_044992', 'ntlop_288038', 'ntlop_290289', 'ntlop_024902', 'ntlop_043608', 'ntlop_024590', 'ntlop_233660', 'ntlop_112458', 'ntlop_116408', 'ntlop_013213', 'ntlop_135352', 'ntlop_283645', 'ntlop_292341', 'ntlop_032611', 'ntlop_281644', 'ntlop_219256', 'ntlop_026102', 'ntlop_246502', 'ntlop_009841', 'ntlop_115201', 'ntlop_000557', 'ntlop_135446', 'ntlop_233189', 'ntlop_021683', 'ntlop_010509', 'ntlop_234869', 'ntlop_012468', 'ntlop_002866', 'ntlop_238847', 'ntlop_233329', 'ntlop_003744', 'ntlop_134549', 'ntlop_023823', 'ntlop_004266', 'ntlop_038987', 'ntlop_012712', 'ntlop_271891', 'ntlop_278256', 'ntlop_036277', 'ntlop_021710', 'ntlop_264814', 'ntlop_211813', 'ntlop_268078', 'ntlop_261249', 'ntlop_253464', 'ntlop_004452', 'ntlop_035399', 'ntlop_119164', 'ntlop_119046', 'ntlop_232042', 'ntlop_021883', 'ntlop_009019', 'ntlop_125441', 'ntlop_013451', 'ntlop_299757', 'ntlop_277881', 'ntlop_237182', 'ntlop_013445', 'ntlop_233121', 'ntlop_219880', 'ntlop_261352', 'ntlop_265471', 'ntlop_024808', 'ntlop_234049', 'ntlop_271874', 'ntlop_295442', 'ntlop_286866', 'ntlop_135239', 'ntlop_288555', 'ntlop_022550', 'ntlop_277892', 'ntlop_264311', 'ntlop_281758', 'ntlop_248183', 'ntlop_219816', 'ntlop_045126', 'ntlop_000832', 'ntlop_125465', 'ntlop_280283', 'ntlop_272752', 'ntlop_013934', 'ntlop_208309', 'ntlop_212210', 'ntlop_234581', 'ntlop_035275', 'ntlop_126524', 'ntlop_001053', 'ntlop_241141', 'ntlop_051138', 'ntlop_121373', 'ntlop_134032', 'ntlop_272566', 'ntlop_126647', 'ntlop_286503', 'ntlop_008505', 'ntlop_300069', 'ntlop_009307', 'ntlop_290603', 'ntlop_002953', 'ntlop_008423', 'ntlop_208744', 'ntlop_021731', 'ntlop_014146', 'ntlop_124266', 'ntlop_010149', 'ntlop_002748', 'ntlop_113385', 'ntlop_021485', 'ntlop_021834', 'ntlop_208311', 'ntlop_043419', 'ntlop_045657', 'ntlop_210437', 'ntlop_009415', 'ntlop_035132', 'ntlop_271347', 'ntlop_008369', 'ntlop_026350', 'ntlop_024964', 'ntlop_003799', 'ntlop_266584', 'ntlop_241375', 'ntlop_124346', 'ntlop_246160', 'ntlop_115563', 'ntlop_021434', 'ntlop_230079', 'ntlop_013359', 'ntlop_210333', 'ntlop_238589', 'ntlop_122989', 'ntlop_288071', 'ntlop_211851', 'ntlop_283646', 'ntlop_232597', 'ntlop_043470', 'ntlop_112287', 'ntlop_232355', 'ntlop_294827', 'ntlop_002873', 'ntlop_290730', 'ntlop_049952', 'ntlop_013433', 'ntlop_218805', 'ntlop_010271', 'ntlop_232347', 'ntlop_231870', 'ntlop_025679', 'ntlop_007766', 'ntlop_285055', 'ntlop_045099', 'ntlop_284851', 'ntlop_273774', 'ntlop_211458', 'ntlop_267590', 'ntlop_033680', 'ntlop_246907', 'ntlop_208703', 'ntlop_227642', 'ntlop_045560', 'ntlop_293409', 'ntlop_280279', 'ntlop_004123', 'ntlop_245493', 'ntlop_118769', 'ntlop_233560', 'ntlop_210627', 'ntlop_036768', 'ntlop_122628', 'ntlop_025090', 'ntlop_284104', 'ntlop_032905', 'ntlop_000822', 'ntlop_035850', 'ntlop_023112', 'ntlop_040513', 'ntlop_261455', 'ntlop_287381', 'ntlop_000193', 'ntlop_041295', 'ntlop_252874', 'ntlop_134591', 'ntlop_122264', 'ntlop_238123', 'ntlop_210259', 'ntlop_122552', 'ntlop_293695', 'ntlop_232585', 'ntlop_041502', 'ntlop_211619', 'ntlop_032423', 'ntlop_232936', 'ntlop_234614', 'ntlop_245455', 'ntlop_230678', 'ntlop_230863', 'ntlop_034559', 'ntlop_115629', 'ntlop_116268', 'ntlop_289075', 'ntlop_208501', 'ntlop_291114', 'ntlop_039411', 'ntlop_246448', 'ntlop_111830', 'ntlop_286747', 'ntlop_210868', 'ntlop_237461', 'ntlop_043293', 'ntlop_264786', 'ntlop_135699', 'ntlop_026225', 'ntlop_121905', 'ntlop_248040', 'ntlop_120777', 'ntlop_010912', 'ntlop_033823', 'ntlop_283786', 'ntlop_042347', 'ntlop_246314', 'ntlop_013030', 'ntlop_299624', 'ntlop_114147', 'ntlop_237883', 'ntlop_023851', 'ntlop_274256', 'ntlop_293179', 'ntlop_289008', 'ntlop_023552', 'ntlop_025397', 'ntlop_284774', 'ntlop_034557', 'ntlop_020618', 'ntlop_233806', 'ntlop_026163', 'ntlop_253694', 'ntlop_273591', 'ntlop_042810', 'ntlop_051174', 'ntlop_134199', 'ntlop_034815', 'ntlop_114105', 'ntlop_123738', 'ntlop_273062', 'ntlop_266538', 'ntlop_125427', 'ntlop_250893', 'ntlop_281816', 'ntlop_008881', 'ntlop_036864', 'ntlop_219668', 'ntlop_135111', 'ntlop_135935', 'ntlop_045349', 'ntlop_116329', 'ntlop_026580', 'ntlop_134001', 'ntlop_114252', 'ntlop_246892', 'ntlop_025168', 'ntlop_292196', 'ntlop_035149', 'ntlop_282783', 'ntlop_272415', 'ntlop_013129', 'ntlop_276142', 'ntlop_114409', 'ntlop_021628', 'ntlop_034901', 'ntlop_040231', 'ntlop_001216', 'ntlop_135222', 'ntlop_011593', 'ntlop_230338', 'ntlop_288291', 'ntlop_002563', 'ntlop_121447', 'ntlop_012922', 'ntlop_131605', 'ntlop_286584', 'ntlop_263847', 'ntlop_276269', 'ntlop_126563', 'ntlop_035472', 'ntlop_034994', 'ntlop_114907', 'ntlop_020727', 'ntlop_000153', 'ntlop_034587', 'ntlop_007849', 'ntlop_235654', 'ntlop_045317', 'ntlop_283683', 'ntlop_253429', 'ntlop_264903', 'ntlop_237651', 'ntlop_124210', 'ntlop_044263', 'ntlop_125561', 'ntlop_238047', 'ntlop_004913', 'ntlop_284415', 'ntlop_008581', 'ntlop_297467', 'ntlop_010024', 'ntlop_209091', 'ntlop_135487', 'ntlop_114488', 'ntlop_294928', 'ntlop_245588', 'ntlop_009677', 'ntlop_292976', 'ntlop_008553', 'ntlop_001450', 'ntlop_009858', 'ntlop_041554', 'ntlop_010958', 'ntlop_296048', 'ntlop_282719', 'ntlop_233308', 'ntlop_121903', 'ntlop_022204', 'ntlop_276356', 'ntlop_126365', 'ntlop_022498', 'ntlop_122496', 'ntlop_294969', 'ntlop_235026', 'ntlop_040608', 'ntlop_264400', 'ntlop_051213', 'ntlop_238681', 'ntlop_025587', 'ntlop_020839', 'ntlop_051282', 'ntlop_271138', 'ntlop_034785', 'ntlop_126893', 'ntlop_136677', 'ntlop_263996', 'ntlop_114953', 'ntlop_210616', 'ntlop_264483', 'ntlop_010668', 'ntlop_234753', 'ntlop_033650', 'ntlop_246279', 'ntlop_035724', 'ntlop_032642', 'ntlop_289031', 'ntlop_126083', 'ntlop_020652', 'ntlop_292657', 'ntlop_265278', 'ntlop_250058', 'ntlop_011521', 'ntlop_245701', 'ntlop_249536', 'ntlop_209083', 'ntlop_298984', 'ntlop_114506', 'ntlop_025349', 'ntlop_024500', 'ntlop_036628', 'ntlop_249686', 'ntlop_122045', 'ntlop_124473', 'ntlop_025264', 'ntlop_008438', 'ntlop_042388', 'ntlop_025411', 'ntlop_210567', 'ntlop_290200', 'ntlop_246881', 'ntlop_209392', 'ntlop_041701', 'ntlop_295955', 'ntlop_218779', 'ntlop_008238', 'ntlop_022664', 'ntlop_273025', 'ntlop_043401', 'ntlop_023678', 'ntlop_008862', 'ntlop_135779', 'ntlop_004733', 'ntlop_040764', 'ntlop_281374', 'ntlop_281637', 'ntlop_278010', 'ntlop_009116', 'ntlop_022234', 'ntlop_234920', 'ntlop_022825', 'ntlop_033979', 'ntlop_288173', 'ntlop_012290', 'ntlop_295364', 'ntlop_252647', 'ntlop_012618', 'ntlop_013536', 'ntlop_265172', 'ntlop_024437', 'ntlop_265669', 'ntlop_021843', 'ntlop_124216', 'ntlop_288894', 'ntlop_212355', 'ntlop_121034', 'ntlop_112513', 'ntlop_249699', 'ntlop_235616', 'ntlop_211378', 'ntlop_010467', 'ntlop_002715', 'ntlop_281859', 'ntlop_272786', 'ntlop_268150', 'ntlop_035863', 'ntlop_124318', 'ntlop_246092', 'ntlop_296284', 'ntlop_234998', 'ntlop_001359', 'ntlop_001329', 'ntlop_284486', 'ntlop_209607', 'ntlop_271272', 'ntlop_245902', 'ntlop_036469', 'ntlop_122635', 'ntlop_231852', 'ntlop_033112', 'ntlop_273237', 'ntlop_114419', 'ntlop_024562', 'ntlop_121066', 'ntlop_209915', 'ntlop_126782', 'ntlop_034704', 'ntlop_120980', 'ntlop_013915', 'ntlop_045139', 'ntlop_233031', 'ntlop_112534', 'ntlop_012343', 'ntlop_126746', 'ntlop_211344', 'ntlop_036041', 'ntlop_266510', 'ntlop_281725', 'ntlop_285073', 'ntlop_234264', 'ntlop_118539', 'ntlop_210562', 'ntlop_035874', 'ntlop_295851', 'ntlop_291832', 'ntlop_263920', 'ntlop_266687', 'ntlop_040103', 'ntlop_026722', 'ntlop_024852', 'ntlop_261242', 'ntlop_234121', 'ntlop_009237', 'ntlop_033249', 'ntlop_208489', 'ntlop_026068', 'ntlop_264138', 'ntlop_021966', 'ntlop_114780', 'ntlop_263815', 'ntlop_044652', 'ntlop_122163', 'ntlop_210964', 'ntlop_212226', 'ntlop_280386', 'ntlop_135598', 'ntlop_272021', 'ntlop_124082', 'ntlop_231871', 'ntlop_114532', 'ntlop_002787', 'ntlop_273240', 'ntlop_010786', 'ntlop_219255', 'ntlop_034534', 'ntlop_001742', 'ntlop_272631', 'ntlop_218790', 'ntlop_209307', 'ntlop_112641', 'ntlop_008533', 'ntlop_115982', 'ntlop_230743', 'ntlop_294618', 'ntlop_291966', 'ntlop_023554', 'ntlop_003781', 'ntlop_123149', 'ntlop_284929', 'ntlop_009396', 'ntlop_001740', 'ntlop_212254', 'ntlop_276245', 'ntlop_020557', 'ntlop_042405', 'ntlop_128216', 'ntlop_121599', 'ntlop_209395', 'ntlop_011621', 'ntlop_233430', 'ntlop_280244', 'ntlop_267555', 'ntlop_124222', 'ntlop_271266', 'ntlop_267608', 'ntlop_264327', 'ntlop_041179', 'ntlop_210624', 'ntlop_235287', 'ntlop_009705', 'ntlop_292004', 'ntlop_035154', 'ntlop_012408', 'ntlop_288129', 'ntlop_113688', 'ntlop_013658', 'ntlop_121845', 'ntlop_232900', 'ntlop_012281', 'ntlop_241529', 'ntlop_295437', 'ntlop_033936', 'ntlop_135754', 'ntlop_281728', 'ntlop_024541', 'ntlop_013494', 'ntlop_261375', 'ntlop_044937', 'ntlop_008231', 'ntlop_238298', 'ntlop_043870', 'ntlop_014115', 'ntlop_007897', 'ntlop_025393', 'ntlop_003795', 'ntlop_208147', 'ntlop_263761', 'ntlop_124648', 'ntlop_284263', 'ntlop_115231', 'ntlop_265180', 'ntlop_033289', 'ntlop_251496', 'ntlop_249776', 'ntlop_007550', 'ntlop_231712', 'ntlop_013095', 'ntlop_008366', 'ntlop_010493', 'ntlop_000225', 'ntlop_013889', 'ntlop_003168', 'ntlop_118580', 'ntlop_283108', 'ntlop_001708', 'ntlop_126630', 'ntlop_035177', 'ntlop_300044', 'ntlop_001166', 'ntlop_292973', 'ntlop_013033', 'ntlop_035444', 'ntlop_116284', 'ntlop_035836', 'ntlop_036034', 'ntlop_022072', 'ntlop_040702', 'ntlop_276023', 'ntlop_032964', 'ntlop_025197', 'ntlop_227860', 'ntlop_252087', 'ntlop_041480', 'ntlop_210734', 'ntlop_045541', 'ntlop_020948', 'ntlop_009405', 'ntlop_042319', 'ntlop_212122', 'ntlop_003661', 'ntlop_126966', 'ntlop_233947', 'ntlop_013863', 'ntlop_245568', 'ntlop_135208', 'ntlop_113387', 'ntlop_289367', 'ntlop_115216', 'ntlop_013754', 'ntlop_128365', 'ntlop_001853', 'ntlop_292864', 'ntlop_123077', 'ntlop_042274', 'ntlop_125448', 'ntlop_281617', 'ntlop_246196', 'ntlop_238447', 'ntlop_112114', 'ntlop_033796', 'ntlop_044406', 'ntlop_233491', 'ntlop_111906', 'ntlop_010706', 'ntlop_241135', 'ntlop_245144', 'ntlop_219433', 'ntlop_013210', 'ntlop_296969', 'ntlop_126785', 'ntlop_044276', 'ntlop_245433', 'ntlop_249535', 'ntlop_208206', 'ntlop_000776', 'ntlop_208945', 'ntlop_004535', 'ntlop_294139', 'ntlop_126651', 'ntlop_265242', 'ntlop_265486', 'ntlop_253912', 'ntlop_238326', 'ntlop_026276', 'ntlop_024375', 'ntlop_024508', 'ntlop_249113', 'ntlop_025782', 'ntlop_012875', 'ntlop_001352', 'ntlop_012276', 'ntlop_115130', 'ntlop_264248', 'ntlop_235367', 'ntlop_004324', 'ntlop_295737', 'ntlop_245015', 'ntlop_126189', 'ntlop_045081', 'ntlop_036109', 'ntlop_040399', 'ntlop_208487', 'ntlop_238809', 'ntlop_261456', 'ntlop_121484', 'ntlop_035919', 'ntlop_273506', 'ntlop_277868', 'ntlop_231662', 'ntlop_043373', 'ntlop_033534', 'ntlop_233395', 'ntlop_267328', 'ntlop_025620', 'ntlop_045790', 'ntlop_289215', 'ntlop_122180', 'ntlop_290228', 'ntlop_003654', 'ntlop_024306', 'ntlop_128012', 'ntlop_123246', 'ntlop_289121', 'ntlop_253541', 'ntlop_021888', 'ntlop_044290', 'ntlop_026697', 'ntlop_285208', 'ntlop_265184', 'ntlop_033382', 'ntlop_045640', 'ntlop_238453', 'ntlop_265161', 'ntlop_035246', 'ntlop_209658', 'ntlop_040249', 'ntlop_123725', 'ntlop_289767', 'ntlop_034886', 'ntlop_131595', 'ntlop_209007', 'ntlop_121240', 'ntlop_238307', 'ntlop_035982', 'ntlop_026588', 'ntlop_263703', 'ntlop_043594', 'ntlop_272244', 'ntlop_266670', 'ntlop_295662', 'ntlop_022111', 'ntlop_007659', 'ntlop_114976', 'ntlop_283248', 'ntlop_276231', 'ntlop_294962', 'ntlop_040100', 'ntlop_247898', 'ntlop_049999', 'ntlop_210914', 'ntlop_219858', 'ntlop_134572', 'ntlop_014221', 'ntlop_250746', 'ntlop_041079', 'ntlop_050193', 'ntlop_042839', 'ntlop_135183', 'ntlop_281627', 'ntlop_265323', 'ntlop_287222', 'ntlop_020758', 'ntlop_237570', 'ntlop_033407', 'ntlop_012968', 'ntlop_211721', 'ntlop_034606', 'ntlop_266634', 'ntlop_040948', 'ntlop_010240', 'ntlop_293734', 'ntlop_209267', 'ntlop_264350', 'ntlop_135569', 'ntlop_238427', 'ntlop_246582', 'ntlop_010756', 'ntlop_002625', 'ntlop_272111', 'ntlop_251008', 'ntlop_281395', 'ntlop_035128', 'ntlop_023817', 'ntlop_280278', 'ntlop_036695', 'ntlop_002449', 'ntlop_025084', 'ntlop_294396', 'ntlop_271553', 'ntlop_010758', 'ntlop_298734', 'ntlop_032638', 'ntlop_126615', 'ntlop_020780', 'ntlop_231219', 'ntlop_231286', 'ntlop_036052', 'ntlop_112009', 'ntlop_013207', 'ntlop_249987', 'ntlop_001232', 'ntlop_001263', 'ntlop_115066', 'ntlop_021799', 'ntlop_219395', 'ntlop_250033', 'ntlop_001374', 'ntlop_040183', 'ntlop_035685', 'ntlop_118945', 'ntlop_012091', 'ntlop_233798', 'ntlop_211203', 'ntlop_126604', 'ntlop_050847', 'ntlop_232068', 'ntlop_033482', 'ntlop_237692', 'ntlop_136421', 'ntlop_271865', 'ntlop_033754', 'ntlop_127449', 'ntlop_286960', 'ntlop_125404', 'ntlop_036137', 'ntlop_009478', 'ntlop_026232', 'ntlop_294306', 'ntlop_235547', 'ntlop_210492', 'ntlop_276504', 'ntlop_235328', 'ntlop_235289', 'ntlop_265681', 'ntlop_023813', 'ntlop_022874', 'ntlop_235679', 'ntlop_219598', 'ntlop_273348', 'ntlop_265675', 'ntlop_285621', 'ntlop_297808', 'ntlop_210093', 'ntlop_246496', 'ntlop_009047', 'ntlop_111929', 'ntlop_003688', 'ntlop_211395', 'ntlop_004738', 'ntlop_209294', 'ntlop_286047', 'ntlop_241138', 'ntlop_035966', 'ntlop_012761', 'ntlop_012023', 'ntlop_122160', 'ntlop_040901', 'ntlop_285083', 'ntlop_294807', 'ntlop_209738', 'ntlop_135545', 'ntlop_035605', 'ntlop_042727', 'ntlop_264608', 'ntlop_233519', 'ntlop_227190', 'ntlop_245364', 'ntlop_241342', 'ntlop_211830', 'ntlop_237447', 'ntlop_286234', 'ntlop_231906', 'ntlop_289382', 'ntlop_003405', 'ntlop_253438', 'ntlop_124647', 'ntlop_246035', 'ntlop_024434', 'ntlop_035676', 'ntlop_235077', 'ntlop_230174', 'ntlop_234854', 'ntlop_024299', 'ntlop_039450', 'ntlop_253118', 'ntlop_294007', 'ntlop_000075', 'ntlop_271960', 'ntlop_272295', 'ntlop_249173', 'ntlop_234894', 'ntlop_208864', 'ntlop_121099', 'ntlop_273648', 'ntlop_132154', 'ntlop_012926', 'ntlop_040583', 'ntlop_285480', 'ntlop_126392', 'ntlop_250502', 'ntlop_209752', 'ntlop_025404', 'ntlop_033999', 'ntlop_003018', 'ntlop_285455', 'ntlop_035749', 'ntlop_290011', 'ntlop_001324', 'ntlop_245349', 'ntlop_237429', 'ntlop_210596', 'ntlop_283262', 'ntlop_234035', 'ntlop_208688', 'ntlop_121646', 'ntlop_292648', 'ntlop_042738', 'ntlop_026178', 'ntlop_238503', 'ntlop_116384', 'ntlop_020729', 'ntlop_123872', 'ntlop_114466', 'ntlop_282782', 'ntlop_280415', 'ntlop_024286', 'ntlop_040137', 'ntlop_210047', 'ntlop_264901', 'ntlop_012729', 'ntlop_276372', 'ntlop_281596', 'ntlop_013246', 'ntlop_298380', 'ntlop_274173', 'ntlop_264310', 'ntlop_246787', 'ntlop_287729', 'ntlop_036272', 'ntlop_235264', 'ntlop_286524', 'ntlop_271075', 'ntlop_209444', 'ntlop_291208', 'ntlop_043377', 'ntlop_235790', 'ntlop_010838', 'ntlop_293460', 'ntlop_289912', 'ntlop_237374', 'ntlop_208467', 'ntlop_012492', 'ntlop_282647', 'ntlop_218239', 'ntlop_115002', 'ntlop_210884', 'ntlop_133945', 'ntlop_001837', 'ntlop_020769', 'ntlop_272449', 'ntlop_266187', 'ntlop_230701', 'ntlop_042398', 'ntlop_281542', 'ntlop_114918', 'ntlop_013135', 'ntlop_290240', 'ntlop_122041', 'ntlop_021472', 'ntlop_283318', 'ntlop_267273', 'ntlop_112156', 'ntlop_131646', 'ntlop_135681', 'ntlop_039223', 'ntlop_219763', 'ntlop_277925', 'ntlop_232766', 'ntlop_114445', 'ntlop_041803', 'ntlop_210328', 'ntlop_003022', 'ntlop_210092', 'ntlop_246748', 'ntlop_245203', 'ntlop_126588', 'ntlop_208470', 'ntlop_020675', 'ntlop_235119', 'ntlop_003981', 'ntlop_237332', 'ntlop_232707', 'ntlop_045477', 'ntlop_280375', 'ntlop_121869', 'ntlop_035775', 'ntlop_041626', 'ntlop_248007', 'ntlop_043875', 'ntlop_126127', 'ntlop_251786', 'ntlop_237144', 'ntlop_042392', 'ntlop_291407', 'ntlop_126535', 'ntlop_285779', 'ntlop_265766', 'ntlop_211380', 'ntlop_293818', 'ntlop_291211', 'ntlop_123859', 'ntlop_211799', 'ntlop_004989', 'ntlop_035852', 'ntlop_238358', 'ntlop_010416', 'ntlop_120806', 'ntlop_008254', 'ntlop_250738', 'ntlop_135741', 'ntlop_010975', 'ntlop_033584', 'ntlop_249046', 'ntlop_009721', 'ntlop_000799', 'ntlop_135387', 'ntlop_050030', 'ntlop_289425', 'ntlop_208239', 'ntlop_121579', 'ntlop_033064', 'ntlop_210545', 'ntlop_039224', 'ntlop_132209', 'ntlop_033009', 'ntlop_023668', 'ntlop_232732', 'ntlop_210267', 'ntlop_237677', 'ntlop_293161', 'ntlop_209423', 'ntlop_246375', 'ntlop_291149', 'ntlop_034485', 'ntlop_288115', 'ntlop_008508', 'ntlop_238800', 'ntlop_265449', 'ntlop_264884', 'ntlop_230765', 'ntlop_000891', 'ntlop_126752', 'ntlop_122592', 'ntlop_246216', 'ntlop_211724', 'ntlop_219140', 'ntlop_235283', 'ntlop_234181', 'ntlop_032899', 'ntlop_127766', 'ntlop_009008', 'ntlop_114984', 'ntlop_134577', 'ntlop_295520', 'ntlop_282740', 'ntlop_025214', 'ntlop_040301', 'ntlop_034669', 'ntlop_208435', 'ntlop_280226', 'ntlop_039021', 'ntlop_248912', 'ntlop_234996', 'ntlop_004223', 'ntlop_035972', 'ntlop_238136', 'ntlop_208973', 'ntlop_114654', 'ntlop_035666', 'ntlop_208601', 'ntlop_035695', 'ntlop_241117', 'ntlop_121556', 'ntlop_135370', 'ntlop_285764', 'ntlop_126216', 'ntlop_296701', 'ntlop_034689', 'ntlop_121769', 'ntlop_235473', 'ntlop_125914', 'ntlop_121788', 'ntlop_042400', 'ntlop_285469', 'ntlop_121780', 'ntlop_267609', 'ntlop_208342', 'ntlop_265548', 'ntlop_284410', 'ntlop_233438', 'ntlop_233887', 'ntlop_264403', 'ntlop_234644', 'ntlop_233594', 'ntlop_004314', 'ntlop_285099', 'ntlop_124520', 'ntlop_288142', 'ntlop_271844', 'ntlop_118779', 'ntlop_112308', 'ntlop_234028', 'ntlop_012844', 'ntlop_276620', 'ntlop_033176', 'ntlop_012436', 'ntlop_002897', 'ntlop_208691', 'ntlop_264428', 'ntlop_126521', 'ntlop_271301', 'ntlop_267373', 'ntlop_136719', 'ntlop_265756', 'ntlop_013309', 'ntlop_009052', 'ntlop_043579', 'ntlop_267568', 'ntlop_114704', 'ntlop_035862', 'ntlop_245887', 'ntlop_232109', 'ntlop_232392', 'ntlop_274345', 'ntlop_025212', 'ntlop_121907', 'ntlop_121254', 'ntlop_045097', 'ntlop_266196', 'ntlop_123740', 'ntlop_288694', 'ntlop_036203', 'ntlop_040528', 'ntlop_022187', 'ntlop_123850', 'ntlop_122615', 'ntlop_124555', 'ntlop_246157', 'ntlop_268525', 'ntlop_042360', 'ntlop_276051', 'ntlop_135138', 'ntlop_271475', 'ntlop_025198', 'ntlop_294448', 'ntlop_114343', 'ntlop_234802', 'ntlop_233168', 'ntlop_233577', 'ntlop_209785', 'ntlop_291186', 'ntlop_116337', 'ntlop_218414', 'ntlop_237628', 'ntlop_033370', 'ntlop_208354', 'ntlop_208465', 'ntlop_241243', 'ntlop_285186', 'ntlop_208427', 'ntlop_264184', 'ntlop_020719', 'ntlop_042413', 'ntlop_124243', 'ntlop_114228', 'ntlop_232327', 'ntlop_045322', 'ntlop_003546', 'ntlop_122452', 'ntlop_281466', 'ntlop_233015', 'ntlop_133951', 'ntlop_044798', 'ntlop_040931', 'ntlop_000688', 'ntlop_023832', 'ntlop_211355', 'ntlop_039221', 'ntlop_001182', 'ntlop_210397', 'ntlop_238379', 'ntlop_128283', 'ntlop_271350', 'ntlop_264166', 'ntlop_126974', 'ntlop_013661', 'ntlop_044231', 'ntlop_124524', 'ntlop_264429', 'ntlop_286462', 'ntlop_041265', 'ntlop_233875', 'ntlop_001162', 'ntlop_283214', 'ntlop_246121', 'ntlop_126106', 'ntlop_020577', 'ntlop_219567', 'ntlop_011549', 'ntlop_238866', 'ntlop_134650', 'ntlop_128366', 'ntlop_232379', 'ntlop_276293', 'ntlop_241165', 'ntlop_040077', 'ntlop_051200', 'ntlop_013046', 'ntlop_034734', 'ntlop_041317', 'ntlop_010914', 'ntlop_025551', 'ntlop_121502', 'ntlop_235170', 'ntlop_043249', 'ntlop_008674', 'ntlop_045428', 'ntlop_208547', 'ntlop_022620', 'ntlop_045367', 'ntlop_292476', 'ntlop_041155', 'ntlop_014077', 'ntlop_126158', 'ntlop_021647', 'ntlop_237852', 'ntlop_042237', 'ntlop_285022', 'ntlop_251859', 'ntlop_208966', 'ntlop_121636', 'ntlop_265103', 'ntlop_023541', 'ntlop_123763', 'ntlop_125403', 'ntlop_040420', 'ntlop_035944', 'ntlop_245536', 'ntlop_003397', 'ntlop_295775', 'ntlop_209229', 'ntlop_289379', 'ntlop_043889', 'ntlop_023868', 'ntlop_009731', 'ntlop_272857', 'ntlop_210390', 'ntlop_001526', 'ntlop_264808', 'ntlop_272225', 'ntlop_041640', 'ntlop_249582', 'ntlop_231818', 'ntlop_264349', 'ntlop_209690', 'ntlop_000931', 'ntlop_020548', 'ntlop_241275', 'ntlop_238875', 'ntlop_288368', 'ntlop_249745', 'ntlop_209118', 'ntlop_232389', 'ntlop_208222', 'ntlop_122090', 'ntlop_246597', 'ntlop_044035', 'ntlop_289002', 'ntlop_293286', 'ntlop_009607', 'ntlop_012280', 'ntlop_283643', 'ntlop_208312', 'ntlop_127557', 'ntlop_112745', 'ntlop_033804', 'ntlop_210408', 'ntlop_001734', 'ntlop_136294', 'ntlop_253260', 'ntlop_211077', 'ntlop_250418', 'ntlop_032386', 'ntlop_041350', 'ntlop_295449', 'ntlop_025341', 'ntlop_034851', 'ntlop_012008', 'ntlop_013333', 'ntlop_040525', 'ntlop_035639', 'ntlop_209494', 'ntlop_286756', 'ntlop_114776', 'ntlop_025098', 'ntlop_024307', 'ntlop_044488', 'ntlop_284477', 'ntlop_280256', 'ntlop_134628', 'ntlop_295434', 'ntlop_265481', 'ntlop_050090', 'ntlop_003399', 'ntlop_245515', 'ntlop_034772', 'ntlop_039075', 'ntlop_290966', 'ntlop_234472', 'ntlop_009658', 'ntlop_235610', 'ntlop_025644', 'ntlop_249233', 'ntlop_210633', 'ntlop_123937', 'ntlop_134021', 'ntlop_245381', 'ntlop_209966', 'ntlop_253327', 'ntlop_114091', 'ntlop_003498', 'ntlop_022192', 'ntlop_235311', 'ntlop_040504', 'ntlop_042567', 'ntlop_238510', 'ntlop_002630', 'ntlop_276242', 'ntlop_218046', 'ntlop_049867', 'ntlop_210143', 'ntlop_000470', 'ntlop_271284', 'ntlop_124454', 'ntlop_051178', 'ntlop_265764', 'ntlop_010043', 'ntlop_035573', 'ntlop_112686', 'ntlop_034995', 'ntlop_044221', 'ntlop_022402', 'ntlop_042675', 'ntlop_209201', 'ntlop_219922', 'ntlop_278276', 'ntlop_208633', 'ntlop_135209', 'ntlop_114087', 'ntlop_265417', 'ntlop_233588', 'ntlop_043355', 'ntlop_026042', 'ntlop_114847', 'ntlop_013624', 'ntlop_238688', 'ntlop_013038', 'ntlop_238086', 'ntlop_210925', 'ntlop_284006', 'ntlop_135080', 'ntlop_116342', 'ntlop_232128', 'ntlop_241530', 'ntlop_246180', 'ntlop_208523', 'ntlop_283129', 'ntlop_244878', 'ntlop_284525', 'ntlop_211140', 'ntlop_289071', 'ntlop_121659', 'ntlop_114175', 'ntlop_022365', 'ntlop_227290', 'ntlop_125869', 'ntlop_009741', 'ntlop_045705', 'ntlop_237770', 'ntlop_271826', 'ntlop_126803', 'ntlop_283340', 'ntlop_238769', 'ntlop_043863', 'ntlop_121913', 'ntlop_035485', 'ntlop_290737', 'ntlop_131880', 'ntlop_040051', 'ntlop_238426', 'ntlop_289979', 'ntlop_034649', 'ntlop_264432', 'ntlop_125375', 'ntlop_119114', 'ntlop_285219', 'ntlop_034017', 'ntlop_115815', 'ntlop_000535', 'ntlop_112638', 'ntlop_287626', 'ntlop_238369', 'ntlop_299600', 'ntlop_002967', 'ntlop_112068', 'ntlop_036081', 'ntlop_288391', 'ntlop_209337', 'ntlop_210994', 'ntlop_210372', 'ntlop_248002', 'ntlop_231797', 'ntlop_274012', 'ntlop_265587', 'ntlop_210353', 'ntlop_227837', 'ntlop_231720', 'ntlop_026137', 'ntlop_022242', 'ntlop_232213', 'ntlop_010889', 'ntlop_287342', 'ntlop_211193', 'ntlop_012230', 'ntlop_124599', 'ntlop_021878', 'ntlop_135009', 'ntlop_277887', 'ntlop_227964', 'ntlop_125829', 'ntlop_127411', 'ntlop_287056', 'ntlop_003719', 'ntlop_210538', 'ntlop_127255', 'ntlop_209616', 'ntlop_253904', 'ntlop_051109', 'ntlop_134617', 'ntlop_035858', 'ntlop_132223', 'ntlop_294548', 'ntlop_286120', 'ntlop_270996', 'ntlop_009446', 'ntlop_283867', 'ntlop_024800', 'ntlop_208796', 'ntlop_003105', 'ntlop_287437', 'ntlop_036711', 'ntlop_230418', 'ntlop_285402', 'ntlop_022697', 'ntlop_241347', 'ntlop_128272', 'ntlop_116060', 'ntlop_041440', 'ntlop_012927', 'ntlop_013435', 'ntlop_230357', 'ntlop_013005', 'ntlop_235130', 'ntlop_249581', 'ntlop_288283', 'ntlop_238314', 'ntlop_013329', 'ntlop_209393', 'ntlop_128387', 'ntlop_296593', 'ntlop_035558', 'ntlop_032622', 'ntlop_290984', 'ntlop_012058', 'ntlop_288437', 'ntlop_041111', 'ntlop_250012', 'ntlop_044191', 'ntlop_231635', 'ntlop_035276', 'ntlop_024906', 'ntlop_126349', 'ntlop_210601', 'ntlop_288000', 'ntlop_253767', 'ntlop_135233', 'ntlop_292628', 'ntlop_294438', 'ntlop_021495', 'ntlop_211563', 'ntlop_004656', 'ntlop_000775', 'ntlop_004837', 'ntlop_008974', 'ntlop_025787', 'ntlop_010560', 'ntlop_219594', 'ntlop_036130', 'ntlop_252732', 'ntlop_040109', 'ntlop_276528', 'ntlop_253224', 'ntlop_264994', 'ntlop_136295', 'ntlop_272035', 'ntlop_000960', 'ntlop_231839', 'ntlop_024369', 'ntlop_245183', 'ntlop_045113', 'ntlop_250443', 'ntlop_251396', 'ntlop_208623', 'ntlop_128437', 'ntlop_001499', 'ntlop_119182', 'ntlop_122668', 'ntlop_267369', 'ntlop_032560', 'ntlop_273180', 'ntlop_296157', 'ntlop_250604', 'ntlop_210514', 'ntlop_252239', 'ntlop_045463', 'ntlop_288700', 'ntlop_294053', 'ntlop_209920', 'ntlop_293638', 'ntlop_115960', 'ntlop_112692', 'ntlop_278302', 'ntlop_211111', 'ntlop_003255', 'ntlop_211388', 'ntlop_237089', 'ntlop_126836', 'ntlop_132115', 'ntlop_281824', 'ntlop_244911', 'ntlop_283035', 'ntlop_237109', 'ntlop_273842', 'ntlop_022765', 'ntlop_134296', 'ntlop_040681', 'ntlop_232796', 'ntlop_285442', 'ntlop_014021', 'ntlop_022040', 'ntlop_002855', 'ntlop_272538', 'ntlop_237008', 'ntlop_010514', 'ntlop_036527', 'ntlop_010366', 'ntlop_211070', 'ntlop_265405', 'ntlop_288456', 'ntlop_264908', 'ntlop_231930', 'ntlop_271512', 'ntlop_035422', 'ntlop_121875', 'ntlop_008818', 'ntlop_004709', 'ntlop_131636', 'ntlop_113632', 'ntlop_033964', 'ntlop_273890', 'ntlop_292179', 'ntlop_045050', 'ntlop_267644', 'ntlop_026576', 'ntlop_004305', 'ntlop_251427', 'ntlop_237270', 'ntlop_118735', 'ntlop_025513', 'ntlop_114205', 'ntlop_265349', 'ntlop_266526', 'ntlop_238410', 'ntlop_036766', 'ntlop_013085', 'ntlop_113533', 'ntlop_233398', 'ntlop_233107', 'ntlop_250035', 'ntlop_020853', 'ntlop_288761', 'ntlop_295537', 'ntlop_233444', 'ntlop_234507', 'ntlop_250129', 'ntlop_120830', 'ntlop_022902', 'ntlop_233229', 'ntlop_009714', 'ntlop_111941', 'ntlop_049938', 'ntlop_012441', 'ntlop_043500', 'ntlop_024410', 'ntlop_128114', 'ntlop_043251', 'ntlop_272452', 'ntlop_112778', 'ntlop_234661', 'ntlop_238668', 'ntlop_248113', 'ntlop_290383', 'ntlop_033057', 'ntlop_290192', 'ntlop_024905', 'ntlop_124332', 'ntlop_121957', 'ntlop_034617', 'ntlop_283659', 'ntlop_232184', 'ntlop_238874', 'ntlop_210099', 'ntlop_245308', 'ntlop_268297', 'ntlop_264244', 'ntlop_013890', 'ntlop_024578', 'ntlop_009316', 'ntlop_044254', 'ntlop_208123', 'ntlop_014263', 'ntlop_010568', 'ntlop_003469', 'ntlop_237218', 'ntlop_210462', 'ntlop_039270', 'ntlop_211386', 'ntlop_040885', 'ntlop_232686', 'ntlop_209334', 'ntlop_246668', 'ntlop_011513', 'ntlop_264470', 'ntlop_263715', 'ntlop_013010', 'ntlop_271279', 'ntlop_272995', 'ntlop_271554', 'ntlop_210236', 'ntlop_232910', 'ntlop_249060', 'ntlop_025144', 'ntlop_114402', 'ntlop_118493', 'ntlop_040555', 'ntlop_121379', 'ntlop_293403', 'ntlop_287777', 'ntlop_051285', 'ntlop_296115', 'ntlop_122320', 'ntlop_227680', 'ntlop_271592', 'ntlop_121761', 'ntlop_294229', 'ntlop_219867', 'ntlop_136857', 'ntlop_121029', 'ntlop_283866', 'ntlop_013160', 'ntlop_208287', 'ntlop_009630', 'ntlop_126721', 'ntlop_235550', 'ntlop_288842', 'ntlop_118558', 'ntlop_292437', 'ntlop_209241', 'ntlop_288493', 'ntlop_280289', 'ntlop_287534', 'ntlop_211435', 'ntlop_040157', 'ntlop_045653', 'ntlop_289494', 'ntlop_025850', 'ntlop_034489', 'ntlop_112389', 'ntlop_042671', 'ntlop_114481', 'ntlop_285273', 'ntlop_246646', 'ntlop_236982', 'ntlop_285351', 'ntlop_022248', 'ntlop_237745', 'ntlop_022354', 'ntlop_036855', 'ntlop_282901', 'ntlop_042356', 'ntlop_113680', 'ntlop_000705', 'ntlop_040339', 'ntlop_045355', 'ntlop_132068', 'ntlop_233522', 'ntlop_245528', 'ntlop_135876', 'ntlop_265091', 'ntlop_010050', 'ntlop_042739', 'ntlop_253914', 'ntlop_235314', 'ntlop_283660', 'ntlop_272428', 'ntlop_232965', 'ntlop_211678', 'ntlop_021815', 'ntlop_001331', 'ntlop_248072', 'ntlop_136199', 'ntlop_208215', 'ntlop_126184', 'ntlop_128199', 'ntlop_267527', 'ntlop_034996', 'ntlop_010851', 'ntlop_045440', 'ntlop_123933', 'ntlop_026503', 'ntlop_266041', 'ntlop_042785', 'ntlop_273729', 'ntlop_291788', 'ntlop_275720', 'ntlop_022325', 'ntlop_227777', 'ntlop_021967', 'ntlop_045527', 'ntlop_299942', 'ntlop_022466', 'ntlop_042492', 'ntlop_012599', 'ntlop_051145', 'ntlop_282937', 'ntlop_032161', 'ntlop_040356', 'ntlop_001298', 'ntlop_232893', 'ntlop_042384', 'ntlop_294072', 'ntlop_231806', 'ntlop_112077', 'ntlop_001440', 'ntlop_292046', 'ntlop_209020', 'ntlop_299669', 'ntlop_134261', 'ntlop_114951', 'ntlop_210706', 'ntlop_232942', 'ntlop_252378', 'ntlop_012331', 'ntlop_232256', 'ntlop_135148', 'ntlop_211348', 'ntlop_124398', 'ntlop_233993', 'ntlop_025603', 'ntlop_234936', 'ntlop_283415', 'ntlop_210250', 'ntlop_209085', 'ntlop_135552', 'ntlop_026654', 'ntlop_035214', 'ntlop_208940', 'ntlop_281727', 'ntlop_238098', 'ntlop_044285', 'ntlop_287565', 'ntlop_296376', 'ntlop_210978', 'ntlop_112733', 'ntlop_211653', 'ntlop_290550', 'ntlop_282913', 'ntlop_266675', 'ntlop_227365', 'ntlop_291358', 'ntlop_286456', 'ntlop_022097', 'ntlop_033448', 'ntlop_033640', 'ntlop_237114', 'ntlop_041630', 'ntlop_264180', 'ntlop_044610', 'ntlop_022332', 'ntlop_293280', 'ntlop_296268', 'ntlop_244865', 'ntlop_273324', 'ntlop_032348', 'ntlop_295850', 'ntlop_126173', 'ntlop_113534', 'ntlop_122616', 'ntlop_293391', 'ntlop_280367', 'ntlop_237912', 'ntlop_009619', 'ntlop_128459', 'ntlop_112020', 'ntlop_045104', 'ntlop_128240', 'ntlop_025741', 'ntlop_041579', 'ntlop_012770', 'ntlop_126235', 'ntlop_273945', 'ntlop_036336', 'ntlop_126263', 'ntlop_023753', 'ntlop_244947', 'ntlop_035501', 'ntlop_245682', 'ntlop_237764', 'ntlop_125220', 'ntlop_124275', 'ntlop_124658', 'ntlop_023558', 'ntlop_210710', 'ntlop_219876', 'ntlop_283809', 'ntlop_226552', 'ntlop_012027', 'ntlop_114568', 'ntlop_280330', 'ntlop_233971', 'ntlop_045387', 'ntlop_266252', 'ntlop_286432', 'ntlop_000605', 'ntlop_000668', 'ntlop_115032', 'ntlop_121589', 'ntlop_287244', 'ntlop_000464', 'ntlop_284791', 'ntlop_000550', 'ntlop_009483', 'ntlop_040383', 'ntlop_265288', 'ntlop_135112', 'ntlop_026468', 'ntlop_290630', 'ntlop_285816', 'ntlop_023837', 'ntlop_284551', 'ntlop_044508', 'ntlop_273530', 'ntlop_025023', 'ntlop_235049', 'ntlop_039091', 'ntlop_020929', 'ntlop_299460', 'ntlop_272385', 'ntlop_033391', 'ntlop_274129', 'ntlop_250570', 'ntlop_032332', 'ntlop_113810', 'ntlop_024752', 'ntlop_231559', 'ntlop_208543', 'ntlop_024777', 'ntlop_218786', 'ntlop_281563', 'ntlop_112230', 'ntlop_263669', 'ntlop_208657', 'ntlop_000259', 'ntlop_125904', 'ntlop_039309', 'ntlop_001388', 'ntlop_287998', 'ntlop_265857', 'ntlop_001812', 'ntlop_264357', 'ntlop_289586', 'ntlop_135215', 'ntlop_209351', 'ntlop_045183', 'ntlop_287376', 'ntlop_211306', 'ntlop_249569', 'ntlop_022494', 'ntlop_043572', 'ntlop_218770', 'ntlop_286697', 'ntlop_007920']
pool = pd.read_parquet(next(p for p in glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)),
                       columns=['id', 'text']).set_index('id')
missing = [i for i in POOL_IDS if i not in pool.index]
assert not missing, missing[:5]
TRAIN = '/kaggle/temp/pool_rows.csv'   # exp053's rows in pool_rows_v17.csv order (labels mixed)
pool.loc[POOL_IDS].reset_index()[['id', 'text']].to_csv(TRAIN, index=False)
print(len(POOL_IDS), 'pool rows', flush=True)
cmd = [sys.executable, '-u', '/kaggle/temp/glmll/run_pipeline.py', '--bin', BIN, '--segs', SEGDIR + '/*.gguf',
       '--csv', TRAIN, '--out', OUT, '--tag', TAG, '--work', WORK, '--seq-batch', '8', '--state-dtype', 'auto',
       '--max-text-tokens', str(MAX_TEXT_TOKENS), '--prefix', PREFIX]
t = time.time()
r = subprocess.run(cmd, env=env)
print(f'{TAG}: exit {r.returncode} after {time.time() - t:.0f} s', flush=True)
subprocess.run(f'ls -la {OUT}; tail -5 {OUT}/{TAG}_run.log', shell=True)
assert r.returncode == 0
